# Étude de cas : ventes et livraisons

Une entreprise vend des produits en ligne (épicerie, frais, boissons, hygiène, maison, électronique, livres, bricolage, collection) et les livre elle-même, depuis ses entrepôts de la région parisienne, avec sa propre flotte de véhicules.

### Les fichiers (`/work/data`)

| Fichier | Contenu |
|---|---|
| `produits.csv` | le catalogue : `produit_id, nom, categorie, sous_categorie, prix, poids_kg` |
| `commandes.csv` | le journal des commandes, une ligne par événement : `commande_id, client_id, horodatage, evenement, livreur_id, detail` |
| `vehicules/<vehicule>/<AAAA-MM>.csv` | la trace GPS de chaque véhicule, **un point par minute** : `timestamp, livreur_id, lon, lat, vitesse_kmh, dans_vehicule, charge_kg` |
| `communes.csv` | les communes de Paris et de la petite couronne et les 20 arrondissements de Paris, avec leur centre : `commune, code_postal, lon, lat` |
| `communes.geojson` | les contours des mêmes communes (propriétés `commune`, `code_postal`) |

### Le journal des commandes

Une commande apparaît plusieurs fois : une ligne par étape.

| `evenement` | Sens | `livreur_id` | `detail` |
|---|---|---|---|
| `commande` | le client passe commande | — | les produits : `P0201x1\|P0033x4` (produit `x` quantité) |
| `chargee` | la commande est chargée dans un véhicule | le livreur | — |
| `livree` | le client signe ; horodatage à la seconde | le livreur | — |
| `echec` | la livraison n'a pas pu se faire | le livreur | le motif |
| `reportee` | chargée mais pas tentée : rapportée à l'entrepôt en fin de service | le livreur | — |
| `retour` | la commande est abandonnée et retournée à l'entrepôt | le livreur | — |

### Les traces GPS

Un livreur conduit un seul véhicule par jour. `dans_vehicule` vaut `false` quand le livreur est descendu du véhicule ; `charge_kg` est le poids transporté.

### Les services

| Adresse | Service |
|---|---|
| http://localhost:8888 | JupyterLab |
| http://localhost:8080 | Master Spark |
| http://localhost:4040 | Spark UI de la session ouverte |
| http://localhost:18080 | Spark History |
| `mongodb://mongo:27017` | MongoDB |

## 1) Charger la donnée

1.1) La session Spark et les fonctions fournies.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

spark = (SparkSession.builder
         .appName("livraisons")
         .master("spark://spark-master:7077")
         .getOrCreate())
DATA = "/work/data"
RESULTATS = "/work/resultats"


def haversine(lon1, lat1, lon2, lat2):
    """Distance en km entre deux points : noms de colonnes ou colonnes, en degrés."""
    lon1, lat1, lon2, lat2 = [F.col(c) if isinstance(c, str) else c for c in (lon1, lat1, lon2, lat2)]
    dlon = F.radians(lon2 - lon1)
    dlat = F.radians(lat2 - lat1)
    a = F.sin(dlat / 2) ** 2 + F.cos(F.radians(lat1)) * F.cos(F.radians(lat2)) * F.sin(dlon / 2) ** 2
    return 2 * 6371.0 * F.asin(F.sqrt(a))

11:34:22 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


1.2) Définissez un schéma explicite (`StructType`) pour chacun des 4 fichiers CSV, puis chargez `produits.csv`, `commandes.csv`, `communes.csv` et **tous** les fichiers GPS en une seule lecture (`vehicules/*/*.csv`). Les horodatages doivent être de type `timestamp`, `dans_vehicule` de type `boolean`, les nombres de type `double`.

Affichez le nombre de lignes de chaque DataFrame.



In [ ]:
schema_produits = T.StructType([
    T.StructField("produit_id", T.StringType(), True),
    T.StructField("nom", T.StringType(), True),
    T.StructField("categorie", T.StringType(), True),
    T.StructField("sous_categorie", T.StringType(), True),
    T.StructField("prix", T.DoubleType(), True),
    T.StructField("poids_kg", T.DoubleType(), True),
])

schema_commandes = T.StructType([
    T.StructField("commande_id", T.StringType(), True),
    T.StructField("client_id", T.StringType(), True),
    T.StructField("horodatage", T.TimestampType(), True),
    T.StructField("evenement", T.StringType(), True),
    T.StructField("livreur_id", T.StringType(), True),
    T.StructField("detail", T.StringType(), True),
])

schema_communes = T.StructType([
    T.StructField("commune", T.StringType(), True),
    T.StructField("code_postal", T.StringType(), True),
    T.StructField("lon", T.DoubleType(), True),
    T.StructField("lat", T.DoubleType(), True),
])

schema_vehicules = T.StructType([
    T.StructField("timestamp", T.TimestampType(), True),
    T.StructField("livreur_id", T.StringType(), True),
    T.StructField("lon", T.DoubleType(), True),
    T.StructField("lat", T.DoubleType(), True),
    T.StructField("vitesse_kmh", T.DoubleType(), True),
    T.StructField("dans_vehicule", T.BooleanType(), True),
    T.StructField("charge_kg", T.DoubleType(), True),
])


# je précise le caractère a escape pour les produits car sinon certains noms de produits du csv font planter le parsing
produits = spark.read.csv(f"{DATA}/produits.csv", header=True, schema=schema_produits, escape='"')
commandes = spark.read.csv(f"{DATA}/commandes.csv", header=True, schema=schema_commandes)
communes = spark.read.csv(f"{DATA}/communes.csv", header=True, schema=schema_communes)
gps = spark.read.csv(f"{DATA}/vehicules/*/*.csv", header=True, schema=schema_vehicules)

# je fais un count simple pour afficher le nombre de lignes de chaque dataframe sans me préocupper des doublons pour l instant
print("produits :", produits.count())
print("commandes :", commandes.count())
print("communes :", communes.count())
print("gps :", gps.count())

produits : 233


commandes : 846535
communes : 142


gps : 5174905


1.3) L'identifiant du véhicule n'est pas une colonne des fichiers GPS : il est dans le chemin du fichier. Ajoutez une colonne `vehicule_id` (fonction `F.input_file_name()`), et une colonne `jour` (la date du point).

In [13]:
gps = (gps
       .withColumn("vehicule_id", F.regexp_extract(F.input_file_name(), r"vehicules/([^/]+)/", 1))
       .withColumn("jour", F.to_date("timestamp")))

# la premiere ligne des 5 premiers vehicules
(gps.groupBy("vehicule_id")
    .agg(F.min("timestamp").alias("timestamp"))
    .withColumn("jour", F.to_date("timestamp"))
    .select("timestamp", "vehicule_id", "jour")
    .orderBy("vehicule_id")
    .show(5, truncate=False))

+-------------------+-----------+----------+
|timestamp          |vehicule_id|jour      |
+-------------------+-----------+----------+
|2025-01-02 07:38:00|V01        |2025-01-02|
|2025-01-02 07:39:00|V02        |2025-01-02|
|2025-01-04 07:45:00|V03        |2025-01-04|
|2025-01-06 07:39:00|V04        |2025-01-06|
|2025-01-06 07:42:00|V05        |2025-01-06|
+-------------------+-----------+----------+
only showing top 5 rows



In [14]:
# je mets en cache les deux dataframes les plus volumineux pour optimiser leurs réutilisations ensuite
gps = gps.cache()
commandes = commandes.cache()

## 2) Vérifier la donnée

2.1) Combien y a-t-il de commandes, de clients, de livreurs et de véhicules ?

In [17]:
commandes.agg(
    F.countDistinct("client_id").alias("nb_clients"),
    F.countDistinct("commande_id").alias("nb_commandes"),
    F.countDistinct("livreur_id").alias("nb_livreurs"),
).show()
gps.agg(F.countDistinct("vehicule_id").alias("nb_vehicules")).show()

+----------+------------+-----------+
|nb_clients|nb_commandes|nb_livreurs|
+----------+------------+-----------+
|      8000|      271617|         90|
+----------+------------+-----------+



+------------+
|nb_vehicules|
+------------+
|          72|
+------------+



2.2) Comptez les événements du journal par type.

In [18]:
commandes.groupBy("evenement").count().orderBy(F.desc("count")).show()


+---------+------+
|evenement| count|
+---------+------+
|  chargee|286862|
| commande|271617|
|   livree|268682|
|    echec| 11220|
| reportee|  6960|
|   retour|  1194|
+---------+------+



2.3) Pour chaque commande, comptez ses événements `chargee`. Combien de commandes ont été chargées 0, 1, 2, 3… fois ?

In [22]:
chargements = commandes.groupBy("commande_id").agg(
    F.count(F.when(F.col("evenement") == "chargee", 1)).alias("nb_chargee")
)

(chargements
    .filter(F.col("nb_chargee") <= 3)
    .groupBy("nb_chargee")
    .agg(F.count("*").alias("nb_commandes"))
    .orderBy("nb_chargee")
    .show())

+----------+------------+
|nb_chargee|nb_commandes|
+----------+------------+
|         0|        1637|
|         1|      254229|
|         2|       14673|
|         3|        1026|
+----------+------------+



2.4) Pour chaque événement `chargee`, trouvez l'événement qui le précède dans la vie de la même commande. Comptez les `chargee` par événement précédent.

In [23]:
# je regroupe les commandes par id dans l ordre chronologique
fenetre = Window.partitionBy("commande_id").orderBy("horodatage")


(commandes
    .withColumn("precedent", F.lag("evenement").over(fenetre))
    .filter(F.col("evenement") == "chargee")
    .groupBy("precedent")
    .count()
    .orderBy(F.desc("count"))
    .show())


+---------+------+
|precedent| count|
+---------+------+
| commande|269980|
|    echec|  9950|
| reportee|  6932|
+---------+------+



*Un rechargement suit toujours un `echec` (nouvelle tentative) ou un `reportee` (commande rapportée en fin de service) : il y a plus de `chargee` que de `commande`.*

2.5) Au 31/12/2025, combien de commandes ne sont pas livrées ? Séparez les commandes abandonnées (événement `retour`) des commandes encore en cours.

In [25]:
# Les commandes qui ont au moins un événement livree
fin_2025 = commandes.filter(F.col("horodatage") < "2026-01-01")

livrees = fin_2025.filter(F.col("evenement") == "livree").select("commande_id").distinct()

# Je retire toutes les lignes de ces commandes
non_livrees = fin_2025.join(livrees, "commande_id", "left_anti")

# Je separe les commandes abandonnées des commandes en cours
nb_non_livrees = non_livrees.select("commande_id").distinct().count()
nb_abandonnees = non_livrees.filter(F.col("evenement") == "retour").select("commande_id").distinct().count()
nb_en_cours = nb_non_livrees - nb_abandonnees

print("non livrées :", nb_non_livrees)
print("abandonnées :", nb_abandonnees)
print("en cours :", nb_en_cours)


non livrées : 2935
abandonnées : 1194
en cours : 1741


## 3) Les lignes de commande

3.1) Les produits d'une commande sont dans la colonne `detail` de l'événement `commande`. Construisez la table `lignes` : une ligne par produit commandé, avec `commande_id`, `client_id`, `horodatage`, `produit_id`, `quantite`.

In [26]:
lignes = (commandes
    .filter(F.col("evenement") == "commande")
    .withColumn("produit", F.explode(F.split(F.col("detail"), r"\|")))
    .withColumn("produit_id", F.split(F.col("produit"), "x").getItem(0))
    .withColumn("quantite", F.split(F.col("produit"), "x").getItem(1).cast("int"))
    .select("commande_id", "client_id", "horodatage", "produit_id", "quantite"))

lignes.show(5, truncate=False)

+-----------+---------+-------------------+----------+--------+
|commande_id|client_id|horodatage         |produit_id|quantite|
+-----------+---------+-------------------+----------+--------+
|C0000001   |CL05658  |2025-01-01 00:03:44|P0037     |5       |
|C0000001   |CL05658  |2025-01-01 00:03:44|P0063     |1       |
|C0000001   |CL05658  |2025-01-01 00:03:44|P0019     |2       |
|C0000001   |CL05658  |2025-01-01 00:03:44|P0067     |1       |
|C0000001   |CL05658  |2025-01-01 00:03:44|P0069     |2       |
+-----------+---------+-------------------+----------+--------+
only showing top 5 rows



3.2) Ajoutez à `lignes` les colonnes du catalogue et le `montant` de chaque ligne. Construisez la table `commandes` : une ligne par commande, avec `montant`, `poids_kg` et `nb_lignes`.

In [27]:
lignes = (lignes
    .join(produits, "produit_id", "left")
    .withColumn("montant", F.col("prix") * F.col("quantite")))

commande_detail = (lignes
    .groupBy("commande_id", "client_id", "horodatage")
    .agg(
        F.round(F.sum("montant"), 2).alias("montant"),
        F.round(F.sum(F.col("poids_kg") * F.col("quantite")), 2).alias("poids_kg"),
        F.count("*").alias("nb_lignes"),
    ))

lignes.show(5, truncate=False)
commande_detail.show(5, truncate=False)


+----------+-----------+---------+-------------------+--------+----------------------+---------+-------------------+----+--------+-------+
|produit_id|commande_id|client_id|horodatage         |quantite|nom                   |categorie|sous_categorie     |prix|poids_kg|montant|
+----------+-----------+---------+-------------------+--------+----------------------+---------+-------------------+----+--------+-------+
|P0037     |C0000001   |CL05658  |2025-01-01 00:03:44|5       |Cookies chocolat 300 g|Épicerie |Biscuits et snacks |2.6 |0.3     |13.0   |
|P0063     |C0000001   |CL05658  |2025-01-01 00:03:44|1       |Lardons fumés 200 g   |Frais    |Viandes et poissons|2.4 |0.2     |2.4    |
|P0019     |C0000001   |CL05658  |2025-01-01 00:03:44|2       |Capsules café x30     |Épicerie |Petit-déjeuner     |8.5 |0.2     |17.0   |
|P0067     |C0000001   |CL05658  |2025-01-01 00:03:44|1       |Bananes 1 kg          |Frais    |Fruits et légumes  |1.9 |1.0     |1.9    |
|P0069     |C0000001   |CL0

+-----------+---------+-------------------+-------+--------+---------+
|commande_id|client_id|horodatage         |montant|poids_kg|nb_lignes|
+-----------+---------+-------------------+-------+--------+---------+
|C0000168   |CL06333  |2025-01-01 13:16:07|209.35 |60.98   |32       |
|C0000469   |CL03079  |2025-01-01 22:39:48|80.9   |14.03   |13       |
|C0000752   |CL00230  |2025-01-02 16:34:53|218.05 |46.52   |31       |
|C0000810   |CL07244  |2025-01-02 18:29:22|239.45 |78.5    |27       |
|C0000855   |CL04907  |2025-01-02 19:48:36|70.9   |15.87   |15       |
+-----------+---------+-------------------+-------+--------+---------+
only showing top 5 rows



## 4) Les entrepôts

4.1) Pour chaque véhicule et chaque jour, gardez le premier et le dernier point GPS. Arrondissez leurs coordonnées à 0,01° et comptez les journées par position arrondie. Combien d'entrepôts y a-t-il ?

In [ ]:
# a retravailler pour optimiser mais pas le temps
# Un groupe correspond à un véhicule pendant une journée.
par_vehicule_jour = Window.partitionBy("vehicule_id", "jour")

# je souhaite regarder pour chaque véhicule, la longitude et latitude de son point de départ et de son point d arrivée
# en fin de journée. Ensuite, je veux regarder par paire long/lat, par jour, combien de véhicules sont passés par ces points
points_classes = (gps
    .withColumn(
        "rang_debut",
        F.row_number().over(par_vehicule_jour.orderBy(F.col("timestamp").asc()))
    )
    .withColumn(
        "rang_fin",
        F.row_number().over(par_vehicule_jour.orderBy(F.col("timestamp").desc()))
    ))

# On garde le premier point et le dernier point de chaque véhicule-jour.
premiers = (points_classes
    .filter(F.col("rang_debut") == 1)
    .select(
        "vehicule_id", "jour",
        F.round("lon", 2).alias("lon_debut"),
        F.round("lat", 2).alias("lat_debut")
    ))

derniers = (points_classes
    .filter(F.col("rang_fin") == 1)
    .select(
        "vehicule_id", "jour",
        F.round("lon", 2).alias("lon_fin"),
        F.round("lat", 2).alias("lat_fin")
    ))

# On met le départ et l'arrivée sur la même ligne pour comparer leurs emplacements.
journees = (premiers
    .join(derniers, ["vehicule_id", "jour"])
    .filter(
        F.col("lon_debut").isNotNull() & F.col("lat_debut").isNotNull()
        & F.col("lon_fin").isNotNull() & F.col("lat_fin").isNotNull()
    )
    .withColumn(
        "meme_emplacement",
        (F.col("lon_debut") == F.col("lon_fin"))
        & (F.col("lat_debut") == F.col("lat_fin"))
    ))

# On transforme chaque journée en deux observations, sans perdre le type du point.
departs = journees.select(
    "vehicule_id", "jour",
    F.struct(F.col("lon_debut").alias("lon"), F.col("lat_debut").alias("lat")).alias("emplacement"),
    F.lit("depart").alias("type_point"),
    "meme_emplacement"
)

arrivees = journees.select(
    "vehicule_id", "jour",
    F.struct(F.col("lon_fin").alias("lon"), F.col("lat_fin").alias("lat")).alias("emplacement"),
    F.lit("arrivee").alias("type_point"),
    "meme_emplacement"
)

extremites = departs.unionByName(arrivees)

# Compter les véhicules-jours par emplacement, en gardant l'indicateur départ = arrivée.
identite_jour = F.struct("vehicule_id", "jour")
frequence_emplacement = (extremites
    .groupBy("emplacement")
    .agg(
        F.countDistinct(identite_jour).alias("nb_vehicule_jours"),
        F.countDistinct(F.when(F.col("type_point") == "depart", identite_jour)).alias("nb_departs"),
        F.countDistinct(F.when(F.col("type_point") == "arrivee", identite_jour)).alias("nb_arrivees"),
        F.countDistinct(F.when(F.col("meme_emplacement"), identite_jour)).alias("nb_jours_depart_arrivee_meme_emplacement")
    )
    .orderBy(F.desc("nb_vehicule_jours")))

frequence_emplacement.show(100, truncate=False)

+-------------+-----------------+----------+-----------+----------------------------------------+
|emplacement  |nb_vehicule_jours|nb_departs|nb_arrivees|nb_jours_depart_arrivee_meme_emplacement|
+-------------+-----------------+----------+-----------+----------------------------------------+
|{2.45, 48.91}|4217             |4217      |4217       |4217                                    |
|{2.28, 48.94}|3208             |3208      |3208       |3208                                    |
|{2.35, 48.75}|2967             |2967      |2967       |2967                                    |
+-------------+-----------------+----------+-----------+----------------------------------------+



4.2) Construisez la table `entrepots` (`entrepot`, `lon`, `lat`) : `E1`, `E2`… par longitude croissante, position = moyenne des premiers points. Construisez la table `vehicule_entrepot` (`vehicule_id`, `entrepot`) et comptez les véhicules par entrepôt.

In [33]:
# Les coordonnées précises des départs, avec la paire arrondie utilisée pour reconnaître le dépôt.
departs_jour = (points_classes
    .filter(F.col("rang_debut") == 1)
    .filter(F.col("lon").isNotNull() & F.col("lat").isNotNull())
    .withColumn("emplacement", F.struct(
        F.round("lon", 2).alias("lon"),
        F.round("lat", 2).alias("lat")
    ))
    .select("vehicule_id", "jour", "lon", "lat", "emplacement"))

# On calcule la position moyenne précise de chaque emplacement fréquent.
positions_entrepots = (departs_jour
    .join(frequence_emplacement.select("emplacement").distinct(), "emplacement")
    .groupBy("emplacement")
    .agg(
        F.avg("lon").alias("lon"),
        F.avg("lat").alias("lat")
    ))

# On attribue E1, E2... en classant les entrepôts par longitude croissante.
ordre_entrepots = Window.orderBy(F.col("lon").asc(), F.col("lat").asc())
entrepots_avec_cle = (positions_entrepots
    .withColumn("numero", F.row_number().over(ordre_entrepots))
    .withColumn("entrepot", F.concat(F.lit("E"), F.col("numero").cast("string"))))

entrepots = entrepots_avec_cle.select("entrepot", "lon", "lat")

# Pour chaque véhicule, son entrepôt habituel est celui de ses départs le plus fréquent.
frequence_departs_vehicule = (departs_jour
    .groupBy("vehicule_id", "emplacement")
    .agg(F.countDistinct("jour").alias("nb_jours")))

ordre_departs_vehicule = Window.partitionBy("vehicule_id").orderBy(
    F.desc("nb_jours"),
    F.asc(F.col("emplacement.lon")),
    F.asc(F.col("emplacement.lat"))
)

vehicule_entrepot = (frequence_departs_vehicule
    .withColumn("rang", F.row_number().over(ordre_departs_vehicule))
    .filter(F.col("rang") == 1)
    .join(entrepots_avec_cle.select("emplacement", "entrepot"), "emplacement")
    .select("vehicule_id", "entrepot"))

nb_vehicules_par_entrepot = (vehicule_entrepot
    .groupBy("entrepot")
    .agg(F.countDistinct("vehicule_id").alias("nb_vehicules"))
    .orderBy("entrepot"))

entrepots.show(truncate=False)
vehicule_entrepot.show(truncate=False)
nb_vehicules_par_entrepot.show(truncate=False)

16:35:39 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
16:35:39 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
16:35:39 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
16:35:39 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
16:35:39 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
16:35:40 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
16:35:40 WARN WindowExec: No Partition Defined for Window operat

+--------+-----------------+------------------+
|entrepot|lon              |lat               |
+--------+-----------------+------------------+
|E1      |2.277068913341649|48.93671290461351 |
|E2      |2.351513594202896|48.747046985844236|
|E3      |2.448959105999522|48.90670164548262 |
+--------+-----------------+------------------+



16:35:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
16:35:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
16:35:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
16:35:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
16:35:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
16:35:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
16:35:58 WARN WindowExec: No Partition Defined for Window operat

+-----------+--------+
|vehicule_id|entrepot|
+-----------+--------+
|V60        |E1      |
|V59        |E1      |
|V58        |E1      |
|V57        |E1      |
|V56        |E1      |
|V55        |E1      |
|V18        |E1      |
|V17        |E1      |
|V16        |E1      |
|V15        |E1      |
|V14        |E1      |
|V13        |E1      |
|V12        |E1      |
|V11        |E1      |
|V10        |E1      |
|V09        |E1      |
|V08        |E1      |
|V07        |E1      |
|V06        |E1      |
|V05        |E1      |
+-----------+--------+
only showing top 20 rows



16:36:14 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
16:36:14 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
16:36:14 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
16:36:14 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
16:36:15 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
16:36:15 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
16:36:17 WARN WindowExec: No Partition Defined for Window operat

+--------+------------+
|entrepot|nb_vehicules|
+--------+------------+
|E1      |24          |
|E2      |24          |
|E3      |24          |
+--------+------------+



## 5) Les livraisons

On découpe la journée de chaque livreur en **livraisons** : une livraison commence quand le livreur descend du véhicule et dure jusqu'à la livraison suivante. Le chargement du matin, les retours à l'entrepôt et la pause déjeuner sont aussi des périodes à pied : la question 5.6 les distingue.

5.1) Ajoutez à `gps` une colonne `changement` : 1 si le livreur vient de descendre du véhicule (point précédent du même livreur, le même jour, `dans_vehicule = true` ; point courant `false`), 0 sinon. Le premier point de la journée vaut 0.

Affichez les lignes où `changement = 1` pour le livreur `L001` le 2025-01-06.

5.2) Ajoutez une colonne `numero_livraison` : la somme cumulée de `changement` pour chaque livreur et chaque jour. Elle vaut 0 au chargement du matin.

Pour `L001` le 2025-01-06, affichez par `numero_livraison` le premier et le dernier timestamp, et le nombre de minutes à pied.

5.3) Construisez la table `livraisons` : une ligne par (`livreur_id`, `jour`, `numero_livraison`), calculée sur les seuls points où le livreur est à pied, avec `vehicule_id`, `debut`, `fin`, `duree_min` (nombre de points) et la position moyenne `lon`, `lat`. Combien de lignes contient-elle ?

5.4) Ajoutez à chaque livraison ses signatures et ses échecs. Un événement `livree` ou `echec` appartient à une livraison quand il a le même livreur, le même jour, et un horodatage compris entre `debut` et `fin`, avec 1 minute de marge de chaque côté : les points GPS sont à la minute, les événements à la seconde.

Construisez la table `livraisons_ev` : les colonnes de `livraisons`, plus `signature` (horodatage de l'événement), `evenement`, `commande_id` et `client_id`. Une ligne par événement ; une livraison sans événement garde une ligne, avec ces 4 colonnes vides.

5.5) Combien d'événements `livree` et `echec` du journal ne sont rattachés à aucune livraison ?

5.6) Toutes les périodes à pied ne sont pas des livraisons. Ajoutez à `livraisons_ev` une colonne `type` :
- `livraison` : au moins un événement ;
- `entrepot` : sinon, position à moins de 300 m d'un entrepôt ;
- `pause` : les autres.

Pour chaque type : nombre de lignes de `livraisons`, durée moyenne, heure moyenne de début.

5.7) Une même livraison peut compter plusieurs événements : plusieurs commandes à la même adresse, remises à pied l'une après l'autre. Combien de livraisons ont 1, 2, 3… événements ? Combien concernent au moins 2 clients différents ?

5.8) Ajoutez une colonne `duree_ev_min` à chaque événement : le temps écoulé depuis l'événement précédent de la même livraison, ou depuis le début de la livraison pour le premier. Quelle est la durée moyenne d'une remise signée ? D'un échec ?

5.9) Pour chaque livreur : nombre de livraisons signées, nombre d'échecs, nombre de jours travaillés, livraisons signées par jour. Affichez les 10 livreurs qui signent le plus de livraisons par jour.

## 6) Les distances

6.1) Ajoutez à `gps` une colonne `distance_km` : la distance depuis le point précédent du même livreur, le même jour (`haversine`). Le premier point de la journée vaut 0.

*Un point par minute coupe les virages : la distance calculée sous-estime d'environ 12 % la distance réellement parcourue.*

6.2) Kilomètres parcourus sur l'année par chaque véhicule : les 10 premiers, et le total de la flotte.

6.3) Kilomètres parcourus sur l'année par chaque livreur : les 10 premiers.

6.4) Kilomètres parcourus par jour : moyenne par véhicule-jour, pour chaque entrepôt.

6.5) Distance entre deux livraisons : la somme de `distance_km` de chaque (`livreur_id`, `jour`, `numero_livraison`), c'est-à-dire le trajet qui suit la livraison jusqu'à la suivante. Moyenne et médiane, pour les livraisons de type `livraison`.

## 7) Les ventes

7.1) Chiffre d'affaires par mois (mois de la commande).

7.2) Panier moyen : montant moyen d'une commande.

7.3) Panier médian.

7.4) Top 10 des produits par chiffre d'affaires.

7.5) Top 10 des produits par quantité vendue.

7.6) Top 10 des clients par dépense sur l'année.

7.7) Top 10 des clients par nombre de commandes.

## 8) La qualité de service

8.1) Délai entre la commande et sa livraison, en jours calendaires (date de la signature − date de la commande). Combien de commandes livrées pour chaque délai ?

8.2) Part des commandes livrées le lendemain, pour chaque mois de commande.

8.3) Construisez la table `suivi` : une ligne par commande, avec `nb_tentatives` (événements `livree` + `echec`), `nb_echecs` et `issue` (`livree`, `retour` ou `en cours`). Combien de commandes pour chaque nombre de tentatives ?

8.4) Motifs d'échec : nombre d'échecs par motif.

8.5) Top 10 des clients au plus fort taux d'échec (échecs / tentatives), parmi les clients qui ont au moins 10 tentatives.

## 9) Les clients

On construit la table `clients` colonne par colonne : une ligne par client qui a passé au moins une commande.

9.1) Construisez la table `clients` : `client_id`, `nb_commandes`, `ca` (somme des montants), `panier_moyen`.

9.2) Ajoutez `premiere_commande` et `derniere_commande` (dates).

9.3) Ajoutez `lon` et `lat` : la position moyenne des livraisons où le client a signé. Combien de clients n'ont aucune position ?

9.4) Ajoutez `commune` et `code_postal` : la commune dont le centre (`communes.csv`) est le plus proche de la position du client.

9.5) Les 10 communes qui passent le plus de commandes, avec leur chiffre d'affaires.

9.6) Ajoutez `tentatives`, `echecs` et `taux_echec` (à partir de `suivi`, 8.3).

9.7) Ajoutez `categorie_dominante` (la catégorie où le client dépense le plus) et `part_dominante` (part de sa dépense dans cette catégorie).

9.8) Ajoutez `unites_par_ligne` : la quantité moyenne par ligne de commande.

9.9) Segments de dépense annuelle (`ca`) : moins de 1 000 €, 1 000 à 5 000 €, 5 000 à 20 000 €, 20 000 à 50 000 €, 50 000 € et plus. Ajoutez la colonne `segment`, puis pour chaque segment : nombre de clients, chiffre d'affaires, chiffre d'affaires moyen par client, part du chiffre d'affaires total.

9.10) Nouveaux clients par mois (mois de la première commande), en tableau et en graphique.

9.11) Pour chaque catégorie dominante : nombre de clients et chiffre d'affaires.

9.12) Clients atypiques : les 10 clients dont `unites_par_ligne` est la plus élevée, avec le rapport à la moyenne de tous les clients, leur `ca` et leur `categorie_dominante`.

## 10) Data visualisation

Échelles de couleur : blanc ou clair pour 0, foncé pour les valeurs élevées.

10.1) Heatmap jour de la semaine × heure du nombre de commandes passées.

10.2) Carte du nombre de livraisons signées par commune ou arrondissement (`communes.geojson`, carte choroplèthe).

10.3) Heatmap heure × mois de la vitesse moyenne quand le livreur est à bord et que le véhicule roule.

10.4) Treemap du chiffre d'affaires : catégorie → sous-catégorie.

## 11) Sauvegarde et modélisation

11.2) **Modélisation.** Les analyses précédentes recalculent les mêmes jointures et agrégations. Proposez des collections MongoDB qui rendent ces questions immédiates : pour chaque collection, ce que représente un document, ses champs, ses tableaux imbriqués, et les questions des parties 5 à 9 qu'elle simplifie. Écrivez votre proposition dans la cellule ci-dessous.

*Votre proposition :*